In [4]:
##### Creates rasters of HA of ag land area (crop + livestock)
### Combines MapSPAM physical area across all crops with grazeland data
# unit = ha

import os
import pandas as pd
import geopandas as gpd
import numpy as np
import rasterio
from pathlib import Path
from rasterio.warp import reproject
from rasterio.enums import Resampling
from rasterio.windows import Window, from_bounds

In [2]:
##### Load data

# Get the current working directory
cd = os.path.dirname(os.getcwd())

# reference raster
ref_path = f"{cd}/Data/Clean/Production/total_production_USD_2020.tif"

# grassland raster
raw_grassland = f"{cd}/Data/Raw/Predictors/Grazeland_Parente/gpw_grassland_rf.med.filt.bthr.reg_c_30m_20200101_20201231_go_epsg.4326_v2.tif"

In [3]:
##### Add MapSPAM values to get total HA of crop production 

# set paths
spam_dir = Path(f"{cd}/Data/Raw/IFPRI_MapSPAM/spam2020V2r0_global_physical_area/")
output_path = f"{cd}/Data/Raw/Predictors/Ag_land_ME/crop_land_ha_2020.tif"

# get all technology rasters 
a_rasters = sorted(spam_dir.glob("spam*_A.tif"))

with rasterio.open(a_rasters[0]) as src:
    meta  = src.meta.copy()
    shape = src.shape

total = np.zeros(shape, dtype=np.float64)

for path in a_rasters:
    with rasterio.open(path) as src:
        data = src.read(1).astype(np.float64)
        data = np.where(np.isnan(data), 0, data)  # nan → 0 before summing
        total += data

meta.update(dtype="float32", count=1, nodata=-9999, compress="lzw")
out_arr = total.astype(np.float32)
out_arr[out_arr == 0] = -9999  # pixels with no cropland → nodata

with rasterio.open(output_path, "w", **meta) as dst:
    dst.write(out_arr, 1)

In [ ]:
##### Get cultivated grassland stats
# in 30m resolution
# assume each cultivated grassland pixel is 100% cultivated grassland 
# so ha in pixel = ha of cultivated grassland (but because of size of raster, taking rough guesstimate approach to ha conversion)
# then resample


### PREP
output_path = f"{cd}/Data/Raw/Predictors/Ag_land_ME/grazeland_land_ha_2020.tif"

# set destination raster
with rasterio.open(ref_path) as ref:
    dst_crs       = ref.crs
    dst_transform = ref.transform
    dst_shape     = ref.shape
    dst_meta      = ref.meta.copy()

dst_meta.update(dtype="float32", count=1, nodata=-9999, compress="lzw")

# set chunks (becuase origional raseter is in 30m resolution)
chunk_rows = 100  

with rasterio.open(raw_grassland) as src, \
     rasterio.open(output_path, "w", **dst_meta) as dst_file:

    n_dst_rows, n_dst_cols = dst_shape

    for row_start in range(0, n_dst_rows, chunk_rows):
        row_end = min(row_start + chunk_rows, n_dst_rows)
        n_rows  = row_end - row_start

        print(f"Processing rows {row_start}–{row_end} of {n_dst_rows}...")

        # lat/lon bounds of this destination chunk
        left   = dst_transform.c
        right  = dst_transform.c + n_dst_cols * dst_transform.a
        top    = dst_transform.f + row_start  * dst_transform.e  # e is negative
        bottom = dst_transform.f + row_end    * dst_transform.e

        # find corresponding source window, add buffer to avoid edge clipping
        src_window = from_bounds(left, bottom, right, top, src.transform)
        src_window = Window(
            max(0, src_window.col_off - 10),
            max(0, src_window.row_off - 10),
            min(src.width  - max(0, src_window.col_off - 10), src_window.width  + 20),
            min(src.height - max(0, src_window.row_off - 10), src_window.height + 20),
        )

        # read source chunk and reclassify: class 1 → 1.0, everything else → 0.0
        src_data = src.read(1, window=src_window)
        src_nodata = src.nodata
        binary = np.where(src_data == 1, 1.0, 0.0).astype(np.float32)
        if src_nodata is not None:
            binary[src_data == src_nodata] = 0.0

        src_win_transform = src.window_transform(src_window)

        # destination chunk
        dst_chunk     = np.zeros((n_rows, n_dst_cols), dtype=np.float32)
        dst_win_transform = rasterio.transform.from_bounds(
            left, bottom, right, top, n_dst_cols, n_rows
        )

        reproject(
            source=binary,
            destination=dst_chunk,
            src_transform=src_win_transform,
            src_crs=src.crs,
            dst_transform=dst_win_transform,
            dst_crs=dst_crs,
            resampling=Resampling.sum,
            src_nodata=None,   # 0 = no grassland, not missing
            dst_nodata=0,
        )

        # convert pixel count → ha (each 30m pixel ≈ 0.09 ha)
        dst_chunk *= 0.09

        dst_file.write(
            dst_chunk, 1,
            window=Window(0, row_start, n_dst_cols, n_rows)
        )


Processing rows 0–100 of 2160...
